# Convolutional Neural Network

### Importing the libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
tf.__version__

'2.19.0'

## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [5]:
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)
training_set = train_datagen.flow_from_directory('Downloads/ML Test/CNN/Malaria cell_images/training set',
                                                 target_size = (64, 64),
                                                 batch_size = 32,
                                                 class_mode = 'binary')

Found 21096 images belonging to 2 classes.


### Preprocessing the Test set

In [7]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('Downloads/ML Test/CNN/Malaria cell_images/test set',
                                            target_size = (64, 64),
                                            batch_size = 32,
                                            class_mode = 'binary')

Found 6462 images belonging to 2 classes.


## Part 2 - Building the CNN

### Initialising the CNN

In [10]:
cnn = tf.keras.models.Sequential()

### Step 1 - Convolution

In [12]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

C:\ProgramData\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Step 2 - Pooling

In [14]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=1))

### Adding a second convolutional layer

In [16]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a third convolutional layer

In [20]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

In [22]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

In [32]:
cnn.add(tf.keras.layers.Dense(units=200, activation='relu'))

### Step 5 - Output Layer

In [35]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the CNN

### Compiling the CNN

In [39]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

### Training the CNN on the Training set and evaluating it on the Test set

In [42]:
from keras.callbacks import EarlyStopping

# Define EarlyStopping
early_stopping = EarlyStopping(
    monitor='val_accuracy',  
    patience=8,   
    restore_best_weights=True
)

cnn.fit(x = training_set, validation_data = test_set, epochs = 30, callbacks=[early_stopping])

C:\ProgramData\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
660/660 ━━━━━━━━━━━━━━━━━━━━ 562s 836ms/step - accuracy: 0.7898 - loss: 0.4103 - val_accuracy: 0.9392 - val_loss: 0.2267
Epoch 2/30
660/660 ━━━━━━━━━━━━━━━━━━━━ 398s 603ms/step - accuracy: 0.9359 - loss: 0.1858 - val_accuracy: 0.9443 - val_loss: 0.1808
Epoch 3/30
660/660 ━━━━━━━━━━━━━━━━━━━━ 387s 586ms/step - accuracy: 0.9490 - loss: 0.1535 - val_accuracy: 0.9615 - val_loss: 0.1412
Epoch 4/30
660/660 ━━━━━━━━━━━━━━━━━━━━ 357s 541ms/step - accuracy: 0.9504 - loss: 0.1437 - val_accuracy: 0.9554 - val_loss: 0.1685
Epoch 5/30
660/660 ━━━━━━━━━━━━━━━━━━━━ 401s 608ms/step - accuracy: 0.9523 - loss: 0.1395 - val_accuracy: 0.9528 - val_loss: 0.1976
Epoch 6/30
660/660 ━━━━━━━━━━━━━━━━━━━━ 397s 601ms/step - accuracy: 0.9550 - loss: 0.1335 - val_accuracy: 0.9601 - val_loss: 0.1478
Epoch 7/30
660/660 ━━━━━━━━━━━━━━━━━━━━ 403s 610ms/step - accuracy: 0.9553 - loss: 0.1308 - val_accuracy: 0.9641 - val_loss: 0.1337
Epoch 8/30
660/660 ━━━━━━━━━━━━━━━━━━━━ 343s 519ms/step - accuracy: 0.9560 -

In [46]:
# Saving the model
cnn.save("Downloads/ML Test/CNN/Malaria cell_images/malaria_model.h5")

# Loading the model
"""
from tensorflow import keras
model = keras.models.load_model("malaria_model.h5")
"""

'\nfrom tensorflow import keras\nmodel = keras.models.load_model("malaria_model.h5")\n'

In [84]:
from joblib import dump, load

# Save model
dump(cnn, "Downloads/ML Test/CNN/Malaria cell_images/my_model.joblib")

# Load model later
# loaded_model = load("my_model.joblib")


['Downloads/ML Test/CNN/Malaria cell_images/my_model.joblib']

## Part 4 - Making a single prediction

In [48]:
training_set.class_indices

{'Parasitized': 0, 'Uninfected': 1}

In [50]:
import numpy as np
from tensorflow.keras.preprocessing import image
import os

test_images = []
images_folder = os.listdir('Downloads/ML Test/CNN/Malaria cell_images/Single Prediction')

for images in images_folder:
    img_path = os.path.join('Downloads/ML Test/CNN/Malaria cell_images/Single Prediction', images)
    img = image.load_img(img_path, target_size=(64, 64))
    img = image.img_to_array(img)
    test_images.append(img)

test_images = np.array(test_images)
results = cnn.predict(test_images)

for i, result in enumerate(results):
    if result[0] == 0:
        prediction = 'Parasitized'
    else:
        prediction = 'Uninfected'
    print(f"Prediction for image {images_folder[i]}: {prediction}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Prediction for image img1.png: Parasitized
Prediction for image img10.png: Parasitized
Prediction for image img11.png: Parasitized
Prediction for image img12.png: Parasitized
Prediction for image img13.png: Uninfected
Prediction for image img14.png: Uninfected
Prediction for image img15.png: Parasitized
Prediction for image img2.png: Uninfected
Prediction for image img3.png: Uninfected
Prediction for image img4.png: Parasitized
Prediction for image img5.png: Parasitized
Prediction for image img6.png: Parasitized
Prediction for image img7.png: Uninfected
Prediction for image img8.png: Uninfected
Prediction for image img9.png: Parasitized
